# Sprint 9 — Feature Engineering
## Beta Bank Customer Churn Prediction

**Goal:** Build a classification model to predict whether a bank customer will leave (churn) using historical behavior data. The project requirement is a minimum **F1 score ≥ 0.59** on the test set.

**Dataset:** `/datasets/Churn.csv` — 10,000 customers, 14 features

**Phases:**
1. Import Libraries & Load Data
2. Explore the Data (EDA)
3. Prepare the Data
4. Examine Class Balance & Train Baseline Models
5. Handle Class Imbalance & Improve Model Performance

---
## Phase 1 — Import Libraries & Load Data

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, recall_score, precision_score
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import resample, shuffle

In [ ]:
# Load the Data
df = pd.read_csv("/datasets/Churn.csv")

---
## Phase 2 — Explore the Data

In [ ]:
# Dataset: Basic Information
print("DATASET SHAPE:", df.shape)
print("\nFirst Few Rows:")
display(df.head())

print("\nDATASET INFO:")
print(df.info())

print("\nSTATISTICAL SUMMARY:")
print(df.describe())

print("\nMISSING VALUES:")
print(df.isnull().sum())

---
## Phase 3 — Prepare the Data

In [ ]:
# PHASE 3. STEP 1: Analyze Missing Values Impact

print("\n" + "="*60)
print("PHASE 3. STEP 1: ANALYZING MISSING VALUES IN TENURE")
print("="*60)

# Compare Distributions
print("\n1. Distribution Comparison:")
print(f"   Tenure with values - Mean: {df['Tenure'].dropna().mean():.2f}, Median: {df['Tenure'].dropna().median():.2f}")
print(f"   Overall Mean (if we used it): {df['Tenure'].mean():.2f}")

# Visualize distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before imputation
df['Tenure'].dropna().hist(bins=20, ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Tenure Distribution (Before Imputation)')
axes[0].set_xlabel('Tenure (years)')
axes[0].axvline(df['Tenure'].dropna().median(), color='red', linestyle='--', label='Median')
axes[0].axvline(df['Tenure'].dropna().mean(), color='green', linestyle='--', label='Mean')
axes[0].legend()

# After imputation (simulate)
tenure_filled = df['Tenure'].fillna(df['Tenure'].median())
tenure_filled.hist(bins=20, ax=axes[1], color='lightcoral', edgecolor='black')
axes[1].set_title('Tenure Distribution (After Median Imputation)')
axes[1].set_xlabel('Tenure (years)')
axes[1].axvline(tenure_filled.median(), color='red', linestyle='--', label='Median')
axes[1].legend()

plt.tight_layout()
plt.show()

# Check if missing values are random
print("\n2. Missing At Random (MAR) Test:")
print("   Comparing churn rates for missing vs non-missing Tenure:")
has_tenure = df['Tenure'].notna()

churn_with_tenure = df[has_tenure]['Exited'].mean()
churn_without_tenure = df[~has_tenure]['Exited'].mean()

print(f"   Churn rate (Tenure present): {churn_with_tenure*100:.1f}%")
print(f"   Churn rate (Tenure missing): {churn_without_tenure*100:.1f}%")
print(f"   Difference: {abs(churn_with_tenure - churn_without_tenure)*100:.1f}%")

if abs(churn_with_tenure - churn_without_tenure) < 0.05:
    print("   Missing values appear random (< 5% difference)")
else:
    print("   Missing values may not be completely random")

print("\n3. Impact Assessment:")
print(f"   - Missing values: {df['Tenure'].isnull().sum()} ({df['Tenure'].isnull().mean()*100:.1f}%)")
print(f"   - Using median (5.0) vs mean ({df['Tenure'].mean():.2f})")
print(f"   - Median is robust to outliers and preserves central tendency")
print(f"   - Distribution shape remains similar after imputation")

print("\nCONCLUSION: Median imputation is appropriate because:")
print("  1. It preserves the distribution's central tendency")
print("  2. It's robust to outliers (some customers have 10-year tenure)")
print("  3. Missing values appear to be random (similar churn rates)")
print("  4. Only 9.1% of data affected - minimal distortion")

In [ ]:
print("\n" + "="*60)
print("DATA PREPARATION")
print("="*60)

# PHASE 3. STEP 1: Handle the missing Tenure values.
print("\nSTEP 1: HANDLING MISSING VALUES")
print(f"   Missing values in Tenure before: {df['Tenure'].isnull().sum()}")

# Fill with median (robust to outliers and makes sense for Tenure)
df['Tenure'].fillna(df['Tenure'].median(), inplace=True)

print(f"   Missing values in Tenure after: {df['Tenure'].isnull().sum()}")
print(f"   Filled with median value: {df['Tenure'].median()}")
print(f"   Rationale: Median is robust and represents typical customer tenure")

In [ ]:
# PHASE 3. STEP 2: Remove Non-Predictive Features
print("\nSTEP 2: REMOVING IRRELEVANT FEATURES")
print(f"   Columns before: {df.shape[1]}")

df_clean = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

print(f"   Columns after: {df_clean.shape[1]}")
print("   Removed: RowNumber, CustomerId, Surname")
print("   Rationale: These are identifiers with no predictive power")

In [ ]:
# PHASE 3. STEP 3: Encode Categorical Variables
print("\nSTEP 3: ENCODING CATEGORICAL VARIABLES")
print(f"   Categorical features: Geography, Gender")

# Check unique values first
print(f"\n   Geography values: {df_clean['Geography'].unique()}")
print(f"   Gender values: {df_clean['Gender'].unique()}")

# Encode Gender (binary: 0/1)
df_clean['Gender'] = df_clean['Gender'].map({'Female': 0, 'Male': 1})

# Encode Geography (one-hot encoding)
df_clean = pd.get_dummies(df_clean, columns=['Geography'], drop_first=True)

print(f"\n   Gender encoded as binary (Female=0, Male=1)")
print(f"   Geography encoded using one-hot encoding")
print(f"   Final columns: {list(df_clean.columns)}")
print(f"   Final shape: {df_clean.shape}")

---
## Phase 4 — Examine Class Balance & Train Baseline Models

In [ ]:
# PHASE 4. STEP 1: Examine the distribution of the target variable (Exited)

print("\n" + "="*60)
print("PHASE 4. STEP 1: EXAMINE TARGET VARIABLE DISTRIBUTION")
print("="*60)

# Separate features and target
X = df_clean.drop('Exited', axis=1)
y = df_clean['Exited']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

print("\nTarget Variable Distribution (Counts):")
print(y.value_counts())

print("\nTarget Variable Distribution (Proportions):")
proportions = y.value_counts(normalize=True)
print(proportions)

# Calculate imbalance ratio
stayed = proportions[0]
churned = proportions[1]
ratio = stayed / churned

print(f"\nClass Imbalance Analysis:")
print(f"   Stayed (0): {stayed*100:.1f}%")
print(f"   Churned (1): {churned*100:.1f}%")
print(f"   Imbalance Ratio: 1:{ratio:.1f}")
print(f"   Significant imbalance - minority class is {ratio:.1f}x smaller!")

# Visualize
y.value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c'])
plt.title('Class Distribution: Customer Churn', fontsize=14, fontweight='bold')
plt.xlabel('Exited (0=Stayed, 1=Churned)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks([0, 1], ['Stayed (0)', 'Churned (1)'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# PHASE 4. STEP 2: Split the data into training, validation, and test sets

print("\n" + "="*60)
print("PHASE 4. STEP 2: SPLIT DATA INTO TRAIN/VALIDATION/TEST SETS")
print("="*60)

# 1st Split: Separate test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTest set separated: {X_test.shape[0]} samples (20%)")

# 2nd Split: Split remaining into train (60%) and validation (20%)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp,
    test_size=0.25,  # 0.25 * 0.8 = 0.2 of total
    random_state=42,
    stratify=y_temp
)

print(f"Training set: {X_train.shape[0]} samples (60%)")
print(f"Validation set: {X_valid.shape[0]} samples (20%)")

print(f"\nFinal Split Summary:")
print(f"  Total samples: {len(X)}")
print(f"  Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Valid: {len(X_valid)} ({len(X_valid)/len(X)*100:.1f}%)")
print(f"  Test:  {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

# Verify class distribution is maintained (stratified split)
print(f"\nClass Distribution Verification (Stratified Split):")
print(f"   Original:   Churned={y.mean()*100:.1f}%")
print(f"   Train set:  Churned={y_train.mean()*100:.1f}%")
print(f"   Valid set:  Churned={y_valid.mean()*100:.1f}%")
print(f"   Test set:   Churned={y_test.mean()*100:.1f}%")
print(f"\n   All sets maintain ~20.4% churn rate - stratification successful!")

In [ ]:
# PHASE 4. STEP 3: Scale the features using StandardScaler

print("\n" + "="*60)
print("PHASE 4. STEP 3: SCALE FEATURES USING STANDARD SCALER")
print("="*60)

# Initialize the scaler
scaler = StandardScaler()

# Fit on training data only, then transform all sets
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

print("\nStandardScaler fitted on training data")
print("All sets transformed using training statistics")

# Verify scaling worked
print(f"\nScaling Verification (Training Set):")
print(f"   Mean (should be ~0): {X_train_scaled.mean():.6f}")
print(f"   Std (should be ~1):  {X_train_scaled.std():.6f}")

# Show example
feature_name = X.columns[0]
print(f"\nExample - '{feature_name}' feature scaling:")
print(f"   Before scaling: mean={X_train[feature_name].mean():.2f}, std={X_train[feature_name].std():.2f}")
print(f"   After scaling:  mean={X_train_scaled[:, 0].mean():.2f}, std={X_train_scaled[:, 0].std():.2f}")

print(f"\nAll features now on same scale (mean=0, std=1)")
print(f"   This prevents features with larger ranges from dominating model")

In [ ]:
# PHASE 4. STEP 4: Train & evaluate baseline models without addressing class imbalance

print("\n" + "="*75)
print("PHASE 4. STEP 4: TRAIN & EVALUATE BASELINE MODELS (NO IMBALANCE HANDLING)")
print("="*75)

print("\nTraining without class imbalance correction to establish baseline performance")

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree':       DecisionTreeClassifier(random_state=42, max_depth=10),
    'Random Forest':       RandomForestClassifier(random_state=42, n_estimators=100)
}

baseline_results = {}

print("\n" + "-"*75)

for name, model in models.items():
    print(f"\n Training {name}...")

    # Train
    model.fit(X_train_scaled, y_train)

    # Predict on validation set
    y_valid_pred  = model.predict(X_valid_scaled)
    y_valid_proba = model.predict_proba(X_valid_scaled)[:, 1]

    # Calculate metrics
    f1      = f1_score(y_valid, y_valid_pred)
    roc_auc = roc_auc_score(y_valid, y_valid_proba)

    # Store results
    baseline_results[name] = {'F1': f1, 'ROC-AUC': roc_auc, 'model': model}

    # Print results
    print(f"   F1 Score:      {f1:.4f}")
    print(f"   ROC-AUC Score: {roc_auc:.4f}")
    print(f"\n   Classification Report:")
    print(classification_report(y_valid, y_valid_pred, target_names=['Stayed', 'Churned']))
    print("-"*60)

# Summary
print("\n BASELINE RESULTS SUMMARY:")
print(f"{'Model':<20} {'F1 Score':<12} {'ROC-AUC':<12}")
print("-"*44)
for name, results in baseline_results.items():
    print(f"{name:<20} {results['F1']:<12.4f} {results['ROC-AUC']:<12.4f}")

print(f"\n Target F1 Score: >=0.59")
print(f"   Best baseline F1: {max(baseline_results.items(), key=lambda x: x[1]['F1'])[1]['F1']:.4f}")

---
## Phase 5 — Handle Class Imbalance & Improve Model Performance

In [ ]:
# PHASE 5. APPROACH 1 — Class Weights

print("\n" + "="*60)
print("PHASE 5. APPROACH 1: CLASS WEIGHTS")
print("="*60)

print("\n Using 'balanced' class weights to penalize minority class misclassification")

# Calculate what 'balanced' means
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

print(f"\nCalculated class weights:")
print(f"   Class 0 (Stayed):  {class_weights[0]:.4f}")
print(f"   Class 1 (Churned): {class_weights[1]:.4f}")
print(f"   Ratio: Churned class weighted {class_weights[1]/class_weights[0]:.2f}x more than Stayed")

# Train Random Forest with class weights
print("\n Training Random Forest with class_weight='balanced'...")

rf_weighted = RandomForestClassifier(
    random_state=42,
    n_estimators=100,
    max_depth=10,
    class_weight='balanced'
)

rf_weighted.fit(X_train_scaled, y_train)

# Predict on validation set
y_valid_pred  = rf_weighted.predict(X_valid_scaled)
y_valid_proba = rf_weighted.predict_proba(X_valid_scaled)[:, 1]

# Metrics
f1_weighted      = f1_score(y_valid, y_valid_pred)
roc_auc_weighted = roc_auc_score(y_valid, y_valid_proba)

print(f"\nRESULTS:")
print(f"   F1 Score:      {f1_weighted:.4f}")
print(f"   ROC-AUC Score: {roc_auc_weighted:.4f}")
print(f"\n   Baseline F1:         0.5573")
print(f"   Improvement:         {f1_weighted - 0.5573:+.4f}")
print(f"   Target (>=0.59):     {'ACHIEVED' if f1_weighted >= 0.59 else 'NOT ACHIEVED'}")

print(f"\n Classification Report:")
print(classification_report(y_valid, y_valid_pred, target_names=['Stayed', 'Churned']))

recall_churned = recall_score(y_valid, y_valid_pred, pos_label=1)
print(f"\nRecall on Churned class: {recall_churned:.4f} (was 0.4300 in baseline)")
print(f"   Improvement: {recall_churned - 0.4300:+.4f}")

In [ ]:
# PHASE 5. APPROACH 2 — Upsampling (Oversampling)

print("\n" + "="*60)
print("PHASE 5. APPROACH 2: UPSAMPLING (OVERSAMPLING)")
print("="*60)

print("\n Balancing classes by replicating minority class samples")

# Combine features and target for resampling
X_train_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_train_df['Exited'] = y_train.values

# Separate majority and minority classes
majority_class = X_train_df[X_train_df['Exited'] == 0]
minority_class = X_train_df[X_train_df['Exited'] == 1]

print(f"\nOriginal class distribution in training set:")
print(f"   Stayed (0):  {len(majority_class)} samples")
print(f"   Churned (1): {len(minority_class)} samples")
print(f"   Ratio: 1:{len(majority_class)/len(minority_class):.2f}")

# Upsample minority class to match majority class
minority_upsampled = resample(
    minority_class,
    replace=True,
    n_samples=len(majority_class),
    random_state=42
)

print(f"\nAfter upsampling:")
print(f"   Stayed (0):  {len(majority_class)} samples")
print(f"   Churned (1): {len(minority_upsampled)} samples")
print(f"   Ratio: 1:1 (perfectly balanced)")

# Combine and shuffle
X_train_balanced    = shuffle(pd.concat([majority_class, minority_upsampled]), random_state=42)
X_train_upsampled   = X_train_balanced.drop('Exited', axis=1).values
y_train_upsampled   = X_train_balanced['Exited'].values

print(f"\nFinal balanced training set: {len(X_train_upsampled)} samples")

# Train Random Forest on upsampled data
print("\n Training Random Forest on balanced (upsampled) data...")

rf_upsampled = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10)
rf_upsampled.fit(X_train_upsampled, y_train_upsampled)

# Predict on validation set (original, not sampled!)
y_valid_pred      = rf_upsampled.predict(X_valid_scaled)
y_valid_proba     = rf_upsampled.predict_proba(X_valid_scaled)[:, 1]

f1_upsampled      = f1_score(y_valid, y_valid_pred)
roc_auc_upsampled = roc_auc_score(y_valid, y_valid_proba)

print(f"\nRESULTS:")
print(f"   F1 Score:      {f1_upsampled:.4f}")
print(f"   ROC-AUC Score: {roc_auc_upsampled:.4f}")
print(f"\n   Baseline F1:  0.5573")
print(f"   Improvement:  {f1_upsampled - 0.5573:+.4f}")
print(f"   Target (>=0.59): {'ACHIEVED' if f1_upsampled >= 0.59 else 'NOT ACHIEVED'}")

print(f"\n Classification Report:")
print(classification_report(y_valid, y_valid_pred, target_names=['Stayed', 'Churned']))

recall_churned = recall_score(y_valid, y_valid_pred, pos_label=1)
print(f"\nRecall on Churned class: {recall_churned:.4f}")

In [ ]:
# PHASE 5. APPROACH 3 — Downsampling (Undersampling)

print("\n" + "="*60)
print("PHASE 5. APPROACH 3: DOWNSAMPLING (UNDERSAMPLING)")
print("="*60)

print("\n Balancing classes by reducing majority class samples")

# Combine features and target for resampling
X_train_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_train_df['Exited'] = y_train.values

majority_class = X_train_df[X_train_df['Exited'] == 0]
minority_class = X_train_df[X_train_df['Exited'] == 1]

print(f"\nOriginal class distribution in training set:")
print(f"   Stayed (0):  {len(majority_class)} samples")
print(f"   Churned (1): {len(minority_class)} samples")
print(f"   Ratio: 1:{len(majority_class)/len(minority_class):.2f}")

# Downsample majority class to match minority class
majority_downsampled = resample(
    majority_class,
    replace=False,
    n_samples=len(minority_class),
    random_state=42
)

print(f"\nAfter downsampling:")
print(f"   Stayed (0):  {len(majority_downsampled)} samples")
print(f"   Churned (1): {len(minority_class)} samples")
print(f"   Ratio: 1:1 (perfectly balanced)")
print(f"   Discarded {len(majority_class) - len(majority_downsampled)} 'Stayed' samples")

# Combine and shuffle
X_train_balanced     = shuffle(pd.concat([majority_downsampled, minority_class]), random_state=42)
X_train_downsampled  = X_train_balanced.drop('Exited', axis=1).values
y_train_downsampled  = X_train_balanced['Exited'].values

print(f"\nFinal balanced training set: {len(X_train_downsampled)} samples")

# Train Random Forest on downsampled data
print("\n Training Random Forest on balanced (downsampled) data...")

rf_downsampled = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10)
rf_downsampled.fit(X_train_downsampled, y_train_downsampled)

# Predict on validation set (original, not downsampled!)
y_valid_pred        = rf_downsampled.predict(X_valid_scaled)
y_valid_proba       = rf_downsampled.predict_proba(X_valid_scaled)[:, 1]

f1_downsampled      = f1_score(y_valid, y_valid_pred)
roc_auc_downsampled = roc_auc_score(y_valid, y_valid_proba)

print(f"\nRESULTS:")
print(f"   F1 Score:      {f1_downsampled:.4f}")
print(f"   ROC-AUC Score: {roc_auc_downsampled:.4f}")
print(f"\n   Baseline F1:  0.5573")
print(f"   Improvement:  {f1_downsampled - 0.5573:+.4f}")
print(f"   Target (>=0.59): {'ACHIEVED' if f1_downsampled >= 0.59 else 'NOT ACHIEVED'}")

print(f"\n Classification Report:")
print(classification_report(y_valid, y_valid_pred, target_names=['Stayed', 'Churned']))

recall_churned = recall_score(y_valid, y_valid_pred, pos_label=1)
print(f"\nRecall on Churned class: {recall_churned:.4f}")

In [ ]:
# PHASE 5: Compare All 3 Approaches

print("\n" + "="*60)
print("PHASE 5. COMPARISON: ALL APPROACHES")
print("="*60)

# Extract metrics
baseline_f1  = baseline_results['Random Forest']['F1']
baseline_roc = baseline_results['Random Forest']['ROC-AUC']

baseline_recall    = recall_score(y_valid, baseline_results['Random Forest']['model'].predict(X_valid_scaled), pos_label=1)
baseline_precision = precision_score(y_valid, baseline_results['Random Forest']['model'].predict(X_valid_scaled), pos_label=1)

weighted_recall    = recall_score(y_valid, rf_weighted.predict(X_valid_scaled), pos_label=1)
weighted_precision = precision_score(y_valid, rf_weighted.predict(X_valid_scaled), pos_label=1)

upsampled_recall    = recall_score(y_valid, rf_upsampled.predict(X_valid_scaled), pos_label=1)
upsampled_precision = precision_score(y_valid, rf_upsampled.predict(X_valid_scaled), pos_label=1)

downsampled_recall    = recall_score(y_valid, rf_downsampled.predict(X_valid_scaled), pos_label=1)
downsampled_precision = precision_score(y_valid, rf_downsampled.predict(X_valid_scaled), pos_label=1)

comparison_data = {
    'Approach':       ['Baseline', 'Class Weights', 'Upsampling', 'Downsampling'],
    'F1 Score':       [baseline_f1, f1_weighted, f1_upsampled, f1_downsampled],
    'ROC-AUC':        [baseline_roc, roc_auc_weighted, roc_auc_upsampled, roc_auc_downsampled],
    'Recall':         [baseline_recall, weighted_recall, upsampled_recall, downsampled_recall],
    'Precision':      [baseline_precision, weighted_precision, upsampled_precision, downsampled_precision],
    'Training Size':  [len(X_train), len(X_train), len(X_train_upsampled), len(X_train_downsampled)]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Find best performer
best_f1_idx       = comparison_df['F1 Score'].idxmax()
best_f1_approach  = comparison_df.loc[best_f1_idx, 'Approach']
best_f1_score     = comparison_df.loc[best_f1_idx, 'F1 Score']

f1_sorted = comparison_df.sort_values('F1 Score', ascending=False)
print(f"\nF1 Score Rankings:")
for _, row in f1_sorted.iterrows():
    status = 'PASSES TARGET' if row['F1 Score'] >= 0.59 else 'Below target'
    print(f"  {row['Approach']}: {row['F1 Score']:.4f} ({status})")

print(f"\nSELECTED MODEL: Random Forest with {best_f1_approach}")
print(f"\nRationale:")
print(f"   1. Highest F1 Score ({best_f1_score:.4f}) — exceeds target by {best_f1_score - 0.59:.4f}")
print(f"   2. Best balance between precision and recall")
print(f"   3. Strong ROC-AUC ({roc_auc_weighted:.4f}) — excellent discrimination")
print(f"   4. Uses all training data efficiently (no data loss)")
print(f"   5. Simple to implement in production")

In [ ]:
# PHASE 5: Hyperparameter Tuning with GridSearchCV

print("\n" + "="*70)
print("PHASE 5: MODEL TUNING")
print("="*70)
print("\nOptimizing Random Forest with Class Weights")
print("Goal: Tune model hyperparameters to maximize F1 score\n")

# Define parameter grid
param_grid = {
    'n_estimators':     [50, 100, 150],
    'max_depth':        [5, 10, 15, 20],
    'min_samples_split':[2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print("Parameter Grid:")
for k, v in param_grid.items():
    print(f"  {k}: {v}")

total_combos = 1
for v in param_grid.values():
    total_combos *= len(v)
print(f"\nTotal combinations to test: {total_combos}")

# Initialize base model with class weights
rf_tuning = RandomForestClassifier(random_state=42, class_weight='balanced')

# GridSearchCV with 3-fold cross-validation
print("\nPerforming GridSearchCV (this may take a few minutes)...")
print("Using 3-fold cross-validation on training data")

grid_search = GridSearchCV(
    estimator=rf_tuning,
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print("\n" + "="*70)
print("GRID SEARCH RESULTS")
print("="*70)

print(f"\nBest Parameters Found:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest Cross-Validation F1 Score: {grid_search.best_score_:.4f}")

# Evaluate tuned model on validation set
rf_tuned            = grid_search.best_estimator_
y_valid_pred_tuned  = rf_tuned.predict(X_valid_scaled)
y_valid_proba_tuned = rf_tuned.predict_proba(X_valid_scaled)[:, 1]

f1_tuned        = f1_score(y_valid, y_valid_pred_tuned)
roc_auc_tuned   = roc_auc_score(y_valid, y_valid_proba_tuned)
recall_tuned    = recall_score(y_valid, y_valid_pred_tuned)
precision_tuned = precision_score(y_valid, y_valid_pred_tuned)

print("\nValidation Set Performance:")
print(f"  F1 Score:  {f1_tuned:.4f}")
print(f"  ROC-AUC:   {roc_auc_tuned:.4f}")
print(f"  Recall:    {recall_tuned:.4f}")
print(f"  Precision: {precision_tuned:.4f}")

print(f"\nCOMPARISON: BEFORE vs AFTER TUNING")
print(f"{'Metric':<20} {'Before Tuning':<15} {'After Tuning':<15} {'Change':<10}")
print("-"*70)
print(f"{'F1 Score':<20} {f1_weighted:<15.4f} {f1_tuned:<15.4f} {f1_tuned - f1_weighted:+.4f}")
print(f"{'ROC-AUC':<20} {roc_auc_weighted:<15.4f} {roc_auc_tuned:<15.4f} {roc_auc_tuned - roc_auc_weighted:+.4f}")
print(f"{'Recall':<20} {weighted_recall:<15.4f} {recall_tuned:<15.4f} {recall_tuned - weighted_recall:+.4f}")
print(f"{'Precision':<20} {weighted_precision:<15.4f} {precision_tuned:<15.4f} {precision_tuned - weighted_precision:+.4f}")

rf_final = rf_tuned

print(f"\nClassification Report (Tuned Model on Validation Set):")
print(classification_report(y_valid, y_valid_pred_tuned, target_names=['Stayed', 'Churned']))
print("\nNext Step: Final Testing on Test Set")

In [ ]:
# PHASE 5. FINAL TESTING: Evaluate Best Model on Test Set

print("\n" + "="*60)
print("PHASE 5. FINAL TESTING: EVALUATE BEST MODEL ON TEST SET")
print("="*60)

print("\nTesting selected model: Random Forest with Class Weights")
print("   Model has NOT seen test set during training or validation")
print("   This provides unbiased estimate of real-world performance\n")

print("Model configuration:")
print(f"   Algorithm:    Random Forest")
print(f"   Class Weight: balanced")
print(f"   n_estimators: 100")
print(f"   max_depth:    10")
print(f"   random_state: 42")

print(f"\nTest set size: {len(X_test)} samples")
print(f"   Class distribution: {y_test.value_counts().to_dict()}")

# Make predictions on test set
print("\nMaking predictions on test set...")
y_test_pred  = rf_weighted.predict(X_test_scaled)
y_test_proba = rf_weighted.predict_proba(X_test_scaled)[:, 1]

# Final metrics
final_f1        = f1_score(y_test, y_test_pred)
final_roc_auc   = roc_auc_score(y_test, y_test_proba)
final_recall    = recall_score(y_test, y_test_pred, pos_label=1)
final_precision = precision_score(y_test, y_test_pred, pos_label=1)

print("\n" + "="*60)
print("FINAL TEST RESULTS")
print("="*60)

print(f"\nPRIMARY METRICS:")
print(f"   F1 Score:  {final_f1:.4f}")
print(f"   ROC-AUC:   {final_roc_auc:.4f}")

print(f"\nPROJECT REQUIREMENT:")
print(f"   Target F1: >=0.59")
print(f"   Status:    {'PASSED' if final_f1 >= 0.59 else 'FAILED'}")
print(f"   Margin:    {final_f1 - 0.59:+.4f}")

print(f"\nDETAILED METRICS:")
print(f"   Precision (Churned): {final_precision:.4f}")
print(f"   Recall (Churned):    {final_recall:.4f}")
print(f"   F1 Score (Churned):  {final_f1:.4f}")

print(f"\nFull Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=['Stayed', 'Churned']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)

print(f"\nCONFUSION MATRIX:")
print(f"   True Negatives  (Stayed correctly):  {cm[0][0]}")
print(f"   False Positives (Stayed -> Churned): {cm[0][1]}")
print(f"   False Negatives (Churned -> Stayed): {cm[1][0]}")
print(f"   True Positives  (Churned correctly): {cm[1][1]}")

# Business impact
total_churners  = cm[1][0] + cm[1][1]
caught_churners = cm[1][1]
missed_churners = cm[1][0]
false_alarms    = cm[0][1]

print(f"\nBUSINESS IMPACT:")
print(f"   Total churners in test set: {total_churners}")
print(f"   Churners caught: {caught_churners} ({caught_churners/total_churners*100:.1f}%)")
print(f"   Churners missed: {missed_churners} ({missed_churners/total_churners*100:.1f}%)")
print(f"   False alarms:    {false_alarms}")

print(f"\nVALIDATION vs TEST COMPARISON:")
print(f"   Validation F1:  0.6381")
print(f"   Test F1:        {final_f1:.4f}")
print(f"   Difference:     {final_f1 - 0.6381:+.4f}")
print(f"   Generalization: {'Model generalizes well' if abs(final_f1 - 0.6381) < 0.05 else 'Possible overfitting'}")

print(f"\n   Validation ROC-AUC: 0.8595")
print(f"   Test ROC-AUC:       {final_roc_auc:.4f}")
print(f"   Difference:         {final_roc_auc - 0.8595:+.4f}")

print("\n" + "="*60)
print("PROJECT COMPLETION STATUS")
print("="*60)

print(f"\nF1 Score >= 0.59:                     {'ACHIEVED' if final_f1 >= 0.59 else 'NOT ACHIEVED'}")
print(f"ROC-AUC Examined:                      YES")
print(f"At least 2 imbalance techniques tested: YES (3 techniques)")
print(f"Proper train/valid/test split:          YES")
print(f"Data preprocessing completed:           YES")
print(f"Baseline model evaluated:               YES")

print(f"\nPROJECT REQUIREMENTS: {'ALL MET' if final_f1 >= 0.59 else 'INCOMPLETE'}")